# ProjectF results

Broker algo wheel, pre-trade cost model and the research handoff. Reads `results/*.json` and the run parquets written by `python -m xcost all`.

In [ ]:
import json, os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))
from xcost import wheel
R = os.path.abspath('../results')
load = lambda n: json.load(open(os.path.join(R, n)))
sig, w, pt, h, pa, models = (load(n) for n in ('signal.json', 'wheel.json', 'pretrade.json', 'handoff.json', 'phaseA_variants.json', 'models.json'))

## 1. The signal

In [ ]:
z = sig['ic_z']; print(f"OOS IC {z['mean']:.4f}, t = {z['t']:.1f}, {z['n_days']} days")
pd.Series(z['by_year']).plot.bar(title='IC by year'); plt.show()
pd.Series(sig['decile_5d_bp']).plot.bar(title='5-day return by decile (bp)'); plt.show()

## 2. The wheel

Three cost benchmarks for the same orders; the oracle column is the paired truth from executing every order with every broker.

In [ ]:
rows = []
for name, r in (('arrival', w['arrival']), ('interval VWAP', w['vwap_schedule_algos']), ('oracle', w['oracle_target'])):
    for g in ('liquid', 'illiquid'):
        for b in ('B', 'C'):
            e = r['by_liquidity'][g][b]; rows.append({'benchmark': name, 'group': g, 'broker': b, 'effect_vs_A': e['bp'], 'ci_lo': e['ci95'][0], 'ci_hi': e['ci95'][1], 'p_best': e['p_best'], 'resid_sd': r['residual_sd_interaction_bp'], 'n_for_1bp': r['power']['1.0bp']})
pd.DataFrame(rows)

In [ ]:
pd.DataFrame(w['vwap_schedule_algos']['strata']).T[['name', 'n', 'mean_pct_adv', 'raw_bp', 'oracle_effect_vs_A', 'oracle_best']]

In [ ]:
orders = pd.read_parquet(os.path.join(R, 'phaseA_gross_wheel_orders.parquet'))
for prefix in ('exec_bp_', 'vwap_bp_'):
    r = wheel.adaptive_allocation(orders, prefix, curves=True)
    c = r['regret_curve']; plt.plot(c['week'], np.array(c['fixed_usd']) / 1e6, label=f'fixed ({prefix})'); plt.plot(c['week'], np.array(c['thompson_usd']) / 1e6, label=f'Thompson ({prefix})')
plt.legend(); plt.ylabel('cumulative regret vs oracle-best routing ($m)'); plt.show()

## 3. Pre-trade cost model

In [ ]:
for key in ('observed', 'oracle'):
    print(key)
    display(pd.DataFrame({k: v['test'] for k, v in pt[key]['specs'].items()}).T)
print('optimiser model', pt['optimiser_model'])
print('contamination', {k: v for k, v in pt['contamination'].items() if not isinstance(v, dict)})

In [ ]:
o = orders.copy(); o['bin'] = pd.qcut(o['pct_adv'], 12, duplicates='drop')
g = o.groupby(['bin', 'broker'], observed=True).agg(x=('pct_adv', 'median'), true=('cost_true_bp', 'mean'), obs=('exec_arrival_bp', 'mean')).reset_index()
for b, gg in g.groupby('broker'):
    plt.plot(gg['x'] * 100, gg['true'], 'o-', label=f'{b} true'); plt.plot(gg['x'] * 100, gg['obs'], 'x:', label=f'{b} observed')
plt.xscale('log'); plt.xlabel('% ADV'); plt.ylabel('bp'); plt.legend(); plt.show()

## 4. The handoff (phase B, out of sample)

In [ ]:
def tab(runs):
    return pd.DataFrame({k: {'paper %/yr': r['paper']['ann_return_bp'] / 100, 'paper SR': r['paper']['sharpe'], 'real %/yr': r['real']['ann_return_bp'] / 100, 'real SR': r['real']['sharpe'], 'survival': r.get('alpha_survival'), 'turnover %/wk': r['turnover_per_week'] * 100, 'IS bp': r['orders']['is_bp'], 'cost %/yr': r['orders']['is_bp_of_capital_per_year'] / 100} for k, r in runs.items()}).T
print('phase A (calibration), gamma* =', h['gamma_star']); display(tab(pa))
print('phase B (handoff)'); display(tab(h['runs']))

In [ ]:
g = h['gamma_star']
for name, ls in (('gross', '-'), (f'cost_g{g}', '-'), ('naive10bp', '-')):
    nav = pd.read_parquet(os.path.join(R, f'phaseB_{name}_nav.parquet'))
    plt.plot(nav.index, (nav['nav_real'] / 1e9 - 1) * 100, label=f'{name} real')
    plt.plot(nav.index, (nav['nav_paper'] / 1e9 - 1) * 100, ':', label=f'{name} paper')
plt.ylabel('% of capital'); plt.legend(); plt.show()